# Chapter 11, Part 4: Graph neural networks and molecular information

An MLP can learn from a fixed descriptor vector. A **graph neural network (GNN)** instead updates atom representations using their neighbors, then combines them into a molecular representation. This notebook builds a small trainable message-passing network using ordinary PyTorch tensors, so its assumptions are visible.

### Learning objectives

- Convert an RDKit molecule into node features, directed edges, and bond features.
- Implement shared message/update functions and a graph-level readout.
- Test atom-permutation **equivariance** (node outputs reorder) and **invariance** (a scalar molecular output stays the same).
- Verify that batching separate molecules does not create communication between them.
- Identify information omitted by an encoding, especially stereochemistry and 3D geometry.

**Scope and runtime:** a handful of embedded molecules, CPU only, one thread, no graph-learning package or pretrained-model download. The initial network has random weights. A single gradient step against a known oxygen count checks differentiability; it is not a property-prediction benchmark. [Part 3](Chapter11_Part3.ipynb) contains the complete measured-data training/validation/test workflow.

**Continue in [Chapter 12](Chapter12_Part1.ipynb)** for a full sequence on graph representations, message-passing architectures, measured-property learning, diagnostics, and 3D geometric networks.

### Start with the molecule, then the tensors

The graph supplies atoms and bonds. The network gives each atom a small list of numbers called a **hidden representation** or **embedding**. A **message** is another calculated list sent along a bond. An **update** combines incoming messages with the atom's current representation. A **readout** collects the final atom representations into one molecular result.

These names describe numerical operations, not physical signals moving along bonds. During training, shared weights learn which combinations are useful for a stated target; the initial random values below have no established chemical meaning.

**Core route:** inspect ethanol's graph → see where information can travel → read one message/update round → inspect the relabeling and pooling figure → understand the stereo collision. The full batching helper and the geometric reflection test are **deeper details**. Chapter 12 develops them further.

**Two symmetry words:** *equivariance* means atom outputs follow an atom reordering; *invariance* means one molecular output stays the same. The figure below makes the distinction visible rather than asking you to memorize the terms.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display
from rdkit import Chem, rdBase
from rdkit.Chem import Draw

OUT = Path("outputs/chapter11_part4")
OUT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(1)
torch.manual_seed(17)
DTYPE = torch.float64  # Tiny arrays; double precision helps numerical comparisons.
print(f"PyTorch {torch.__version__}; RDKit {rdBase.rdkitVersion}; device: CPU")

## 11.4.1. Encode a graph, not an atom-index sequence

For each heavy atom, use four deliberately simple features: atomic number divided by 10, attached hydrogen count divided by 4, formal charge, and aromaticity. These fixed numerical scale factors are definitions, not statistics fitted to data. We encode single, double, triple, and aromatic bonds with four indicator features. Each undirected bond becomes two directed edges so both atoms can receive messages.

This is a **limited teaching encoding**, not an optimized chemical representation. Atomic number as a scalar does not guarantee sensible extrapolation to unseen elements. We omit chirality, bond stereochemistry, isotopes, coordinates, and experimental conditions. Adding explicit hydrogen nodes would require a consistent alternative hydrogen policy.

The atom index is an address for connecting tensors; it is not a chemical property to feed into the network. Source for the molecular API: [RDKit molecule and atom objects](https://www.rdkit.org/docs/source/rdkit.Chem.rdchem.html).

In [ ]:
NODE_FEATURES = ["atomic_number/10", "total_H/4", "formal_charge", "is_aromatic"]
BOND_TYPES = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE,
              Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]

def pack_graphs(molecules):
    if not molecules or any(mol is None or mol.GetNumAtoms() == 0 for mol in molecules):
        raise ValueError("Supply a nonempty list of parsed molecules.")
    nodes, edges, bond_features, graph_ids = [], [], [], []
    offset = 0
    for graph_id, mol in enumerate(molecules):
        for atom in mol.GetAtoms():
            if atom.GetAtomicNum() == 1:
                raise ValueError("This encoding uses implicit H counts, not explicit H nodes.")
            nodes.append([atom.GetAtomicNum()/10, atom.GetTotalNumHs()/4,
                          atom.GetFormalCharge(), float(atom.GetIsAromatic())])
            graph_ids.append(graph_id)
        for bond in mol.GetBonds():
            if bond.GetBondType() not in BOND_TYPES:
                raise ValueError("This lesson supports single/double/triple/aromatic bonds.")
            feature = [float(bond.GetBondType() == kind) for kind in BOND_TYPES]
            a, b = bond.GetBeginAtomIdx()+offset, bond.GetEndAtomIdx()+offset
            edges.extend([(a, b), (b, a)])
            bond_features.extend([feature, feature])
        offset += mol.GetNumAtoms()
    batch = {"nodes": torch.tensor(nodes, dtype=DTYPE),
             "edge_index": torch.tensor(edges, dtype=torch.long).reshape(-1, 2).T,
             "bonds": torch.tensor(bond_features, dtype=DTYPE).reshape(-1, 4),
             "graph_ids": torch.tensor(graph_ids, dtype=torch.long),
             "n_graphs": len(molecules)}
    src, dst = batch["edge_index"]
    assert torch.equal(batch["graph_ids"][src], batch["graph_ids"][dst])
    return batch

smiles = ["CCO", "CC(=O)O", "CCN", "c1ccccc1", "O"]
molecules = [Chem.MolFromSmiles(s) for s in smiles]
batch = pack_graphs(molecules)
display(Draw.MolsToGridImage(molecules, legends=smiles, molsPerRow=3, subImgSize=(230, 160)))
display(pd.DataFrame(batch["nodes"].numpy(), columns=NODE_FEATURES).assign(
    graph_id=batch["graph_ids"].numpy()).head(8))
print("Node matrix:", tuple(batch["nodes"].shape), "directed edges:", batch["edge_index"].shape[1])

### Follow information from one atom

Choose ethanol's oxygen as a reference and count **graph bonds**, not angstroms, along the shortest path to each atom. A local update can bring information one more bond away in each round. The blue region below shows where information originating at oxygen **can** have reached by that round; it is not a learned importance score or a claim that the network uses every possible path.

**Predict:** how many rounds are needed before the oxygen can influence the carbon at the other end of ethanol? Why does summing all atom vectors at the end not make each atom's individual receptive field global?

In [ ]:
import matplotlib.pyplot as plt

flow_molecule = molecules[0]
oxygen_index = next(atom.GetIdx() for atom in flow_molecule.GetAtoms() if atom.GetAtomicNum() == 8)
graph_distances = Chem.GetDistanceMatrix(flow_molecule)[oxygen_index].astype(int)
flow_positions = np.array([[0.0, 0.0], [1.0, 0.35], [2.0, 0.0]])
fig, axes = plt.subplots(1, 3, figsize=(10, 3), layout="constrained")
for rounds, ax in enumerate(axes):
    reached = graph_distances <= rounds
    for bond in flow_molecule.GetBonds():
        pair = flow_positions[[bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()]]
        ax.plot(*pair.T, color="0.5", linewidth=2, zorder=1)
    for index, atom in enumerate(flow_molecule.GetAtoms()):
        ax.scatter(*flow_positions[index], s=700, color="#2563eb" if reached[index] else "#e5e7eb", zorder=2)
        ax.text(*flow_positions[index], f"{atom.GetSymbol()}{index}", ha="center", va="center",
                color="white" if reached[index] else "#334155", zorder=3)
    ax.set(title=f"After {rounds} local round(s)", xlim=(-0.4, 2.4), ylim=(-0.5, 0.85), aspect="equal")
    ax.axis("off")
fig.suptitle("Possible reach of oxygen information in the heavy-atom bond graph")
plt.show()
assert graph_distances.tolist() == [2, 1, 0]
display(pd.DataFrame({"atom_index": range(3), "graph_bonds_from_oxygen": graph_distances}))

**Explain:** two rounds are needed to reach the terminal carbon. Graph readout combines different local summaries into one molecular vector; it does not retroactively send that pooled information back to each atom in this architecture. Initial atom features may themselves summarize chemical information, so the round count is a communication limit in addition to the information already supplied.

## 11.4.2. Shared messages, shared updates, and a readout

Let $h_i^{(0)}$ be an embedding of atom features $x_i$. At each of two rounds,

$$m_i^{(t)}=\sum_{j\in\mathcal N(i)}\tanh\!\left(W_m^{(t)}[h_j^{(t)},e_{ji}]+b_m^{(t)}\right),$$
$$h_i^{(t+1)}=\tanh\!\left(W_u^{(t)}[h_i^{(t)},m_i^{(t)}]+b_u^{(t)}\right).$$

The brackets mean feature concatenation. The same parameters are used for every atom or edge **within a round**; our two rounds have distinct parameter sets. Sum the final atom vectors within each molecule and apply a linear readout.

Neighbor summation is independent of neighbor order, and all atoms use the same functions. Thus, relabeling atoms reorders the node vectors but leaves the graph-level result unchanged, up to floating-point summation differences. This is one small instance of the [message-passing framework of Gilmer et al.](https://proceedings.mlr.press/v70/gilmer17a.html), not a reproduction of that paper's benchmark model.

In [ ]:
class TinyMPNN(nn.Module):
    def __init__(self, hidden=8, rounds=2):
        super().__init__()
        self.embed = nn.Linear(4, hidden)
        self.messages = nn.ModuleList([nn.Linear(hidden+4, hidden) for _ in range(rounds)])
        self.updates = nn.ModuleList([nn.Linear(2*hidden, hidden) for _ in range(rounds)])
        self.readout = nn.Linear(hidden, 1)

    def forward(self, graph):
        h = torch.tanh(self.embed(graph["nodes"]))
        src, dst = graph["edge_index"]
        for message_layer, update_layer in zip(self.messages, self.updates):
            messages = torch.tanh(message_layer(torch.cat([h[src], graph["bonds"]], dim=1)))
            aggregated = torch.zeros_like(h).index_add(0, dst, messages)
            h = torch.tanh(update_layer(torch.cat([h, aggregated], dim=1)))
        pooled = h.new_zeros((graph["n_graphs"], h.shape[1])).index_add(0, graph["graph_ids"], h)
        return self.readout(pooled).squeeze(-1), h, pooled

model = TinyMPNN().to(dtype=DTYPE, device="cpu")
model.eval()
with torch.inference_mode():
    initial_scores, node_embeddings, pooled_embeddings = model(batch)
assert initial_scores.shape == (len(molecules),) and torch.isfinite(initial_scores).all()
display(pd.DataFrame({"SMILES": smiles, "untrained_arbitrary_score": initial_scores.numpy()}))
print("Trainable parameters:", sum(parameter.numel() for parameter in model.parameters()))

`index_add` adds each message to its destination row, then adds each atom vector to its molecule's row. There are no padded atoms or dense adjacency matrices here. The single-atom water graph has zero heavy-atom edges but still has an atom embedding and a valid molecular output. See [PyTorch indexed addition](https://docs.pytorch.org/docs/2.11/generated/torch.Tensor.index_add_.html).

The scores above are **not solubilities, energies, or probabilities**. No target has trained these weights. An invariant output can still be scientifically meaningless.

## 11.4.3. Test atom relabeling and disconnected batching

`Chem.RenumberAtoms(mol, order)` creates a copy whose new atom $i$ is the original atom `order[i]`. We check both the scalar score and the correspondence of all node embeddings. We also compare processing molecules in one disconnected batch with processing them individually.

In [ ]:
ethanol = molecules[0]
permutation = [2, 0, 1]  # New atom positions refer to these original atoms.
renumbered = Chem.RenumberAtoms(ethanol, permutation)
with torch.inference_mode():
    score_a, hidden_a, pooled_a = model(pack_graphs([ethanol]))
    score_b, hidden_b, pooled_b = model(pack_graphs([renumbered]))
    separate_scores = torch.cat([model(pack_graphs([mol]))[0] for mol in molecules])
    reversed_scores = model(pack_graphs(list(reversed(molecules))))[0]
torch.testing.assert_close(score_a, score_b, atol=1e-12, rtol=1e-12)
torch.testing.assert_close(hidden_b, hidden_a[permutation], atol=1e-12, rtol=1e-12)
torch.testing.assert_close(initial_scores, separate_scores, atol=1e-12, rtol=1e-12)
torch.testing.assert_close(reversed_scores, initial_scores.flip(0), atol=1e-12, rtol=1e-12)
print("Atom-permutation score difference:", abs(float(score_a[0]-score_b[0])))
print("Node equivariance, graph invariance, and separate-vs-batched outputs agree.")

# A position-weighted atom-number sum depends on an arbitrary serialization order.
def index_weighted_sum(mol):
    return sum((i+1)*atom.GetAtomicNum() for i, atom in enumerate(mol.GetAtoms()))
assert index_weighted_sum(ethanol) != index_weighted_sum(renumbered)
print("Index-dependent counterexample:", index_weighted_sum(ethanol), index_weighted_sum(renumbered))

### See equivariance and invariance in the actual computed tensors

Each heatmap row below is an atom; each column is one of eight hidden features. Renumbering ethanol changes row order, not the features associated with corresponding atoms. The right-hand plot sums each column over atoms: the two molecular vectors overlap.

These are the same **untrained numerical embeddings** just tested. Their signs and colors are not atom importance or chemical property values. **Predict:** which original row should become the first row after the permutation `[2, 0, 1]`?

In [ ]:
embedding_a, embedding_b = hidden_a.numpy(), hidden_b.numpy()
embedding_limit = max(np.abs(embedding_a).max(), np.abs(embedding_b).max())
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), width_ratios=[1, 1, 1.25], layout="constrained")
for ax, values, labels, title in [(axes[0], embedding_a, ["C0", "C1", "O2"], "Original atom order"),
                                 (axes[1], embedding_b, ["O2", "C0", "C1"], "Renumbered: same atoms")]:
    ax.imshow(values, cmap="RdBu_r", vmin=-embedding_limit, vmax=embedding_limit, aspect="auto")
    ax.set(yticks=range(3), yticklabels=labels, xticks=range(8),
           xlabel="Hidden feature index", title=title)
axes[2].plot(range(8), pooled_a.numpy()[0], "o-", label="Original sum", color="#2563eb")
axes[2].plot(range(8), pooled_b.numpy()[0], "x--", label="Renumbered sum", color="#b45309")
axes[2].set(xlabel="Hidden feature index", ylabel="Pooled arbitrary feature value",
            title="One molecular vector is unchanged")
axes[2].legend(fontsize=8)
plt.show()
np.testing.assert_allclose(embedding_b, embedding_a[permutation], atol=1e-12)
np.testing.assert_allclose(embedding_a.sum(axis=0), pooled_a.numpy()[0], atol=1e-12)

**Research use:** this is a structural software check to run before trusting a graph-property study. If arbitrary atom renumbering changes a purported scalar molecular property, investigate the encoding or architecture. Passing this check does not establish predictive accuracy; the measured-data splits and baselines from Part 3 remain necessary.

These checks verify the intended architecture for the examples; they are not a proof that all distinct molecular graphs receive different embeddings. With two rounds, an atom can receive graph information from at most two bonds away, in addition to whatever local information its initial features already contain. Global pooling combines these local summaries, but does not automatically identify every long-range arrangement.

**Sum versus mean:** sum pooling retains information about graph size, while mean pooling normalizes by atom count. This is a design choice, not a guarantee of physical extensivity or intensivity. For this model, graph-local messages and sum pooling make the *pooled vector* additive across disconnected components. A biased final readout does not generally preserve exact output additivity. A physical energy model needs the appropriate structure and validation.

## 11.4.4. Check differentiability with one gradient step

For a numerical check only, use the exactly known **number of oxygen atoms** in each input graph. Direct counting is already an exact baseline; a GNN is unnecessary for this task. We make one small full-batch gradient-descent step and confirm that its loss decreases. This checks that the graph operations connect to trainable parameters. It does not establish generalization and uses no experimental labels or held-out evaluation.

In [ ]:
oxygen_counts = torch.tensor([sum(a.GetAtomicNum() == 8 for a in mol.GetAtoms())
                              for mol in molecules], dtype=DTYPE)
model.train()
model.zero_grad(set_to_none=True)
scores, _, _ = model(batch)
loss_before = nn.functional.mse_loss(scores, oxygen_counts)
loss_before.backward()
gradient_norm = torch.sqrt(sum(parameter.grad.square().sum() for parameter in model.parameters()))
assert torch.isfinite(gradient_norm) and float(gradient_norm) > 0
learning_rate = 1e-3
with torch.no_grad():
    for parameter in model.parameters():
        parameter.add_(parameter.grad, alpha=-learning_rate)
model.eval()
with torch.inference_mode():
    loss_after = nn.functional.mse_loss(model(batch)[0], oxygen_counts)
    updated_a = model(pack_graphs([ethanol]))[0]
    updated_b = model(pack_graphs([renumbered]))[0]
assert float(loss_after) < float(loss_before.detach())
torch.testing.assert_close(updated_a, updated_b, atol=1e-12, rtol=1e-12)
print(f"Oxygen-count diagnostic MSE: {float(loss_before.detach()):.6f} -> {float(loss_after):.6f}")
print(f"Gradient norm: {float(gradient_norm):.6f}; exact counting baseline MSE: 0")

## 11.4.5. A model cannot recover distinctions its input erased

Consider two lactic-acid enantiomers. Their graph connectivity, atomic numbers, formal charges, attached H counts, and bond types agree under atom correspondence. Our encoding omits the stereochemical tags, so these inputs produce identical tensors and identical outputs for **any** weights in this architecture.

This does not imply that all properties of enantiomers are equal. An endpoint involving a chiral environment can distinguish them. To model such a distinction, the representation and experimental context must include relevant information. RDKit stores stereo tags; this particular feature function simply does not use them.

In [ ]:
stereo_smiles = ["C[C@H](O)C(=O)O", "C[C@@H](O)C(=O)O"]
enantiomers = [Chem.MolFromSmiles(s) for s in stereo_smiles]
left, right = [pack_graphs([mol]) for mol in enantiomers]
assert Chem.MolToSmiles(enantiomers[0]) != Chem.MolToSmiles(enantiomers[1])
for key in ("nodes", "edge_index", "bonds", "graph_ids"):
    assert torch.equal(left[key], right[key])
with torch.inference_mode():
    stereo_scores = model(pack_graphs(enantiomers))[0]
torch.testing.assert_close(stereo_scores[0], stereo_scores[1], atol=1e-12, rtol=1e-12)
display(Draw.MolsToGridImage(enantiomers, legends=["Enantiomer 1", "Enantiomer 2"],
                           molsPerRow=2, subImgSize=(310, 200)))
print("Different isomeric SMILES, identical encoded inputs and scores:", stereo_scores.tolist())

### Geometry and symmetry are separate design choices

A 2D bond graph does not identify a conformer. A 3D model can use distances, angles, or vector features; its output should transform appropriately under rotations and translations. A scalar energy should be invariant, while a predicted force vector should rotate with the molecule. Three-dimensional models such as [DimeNet](https://arxiv.org/abs/2003.03123) explicitly incorporate geometric information.

Pairwise distances alone also cannot distinguish mirror-reflected point arrangements. The following **geometric example is not a molecular conformer**: four labeled noncoplanar points and their reflection have equal distance matrices and opposite signed orientation. An architecture receiving only these distances cannot distinguish the pair. Handedness requires additional suitable information and symmetry choices; distances do not make every 3D model chirality-aware.

In [ ]:
points = np.array([[0., 0., 0.], [1., 0., 0.], [0., 2., 0.], [0., 0., 3.]])
reflected = points * [-1, 1, 1]
def distance_matrix(x):
    return np.linalg.norm(x[:, None, :] - x[None, :, :], axis=-1)
def signed_orientation(x):
    return np.linalg.det(x[1:] - x[0])
np.testing.assert_allclose(distance_matrix(points), distance_matrix(reflected))
assert signed_orientation(points) * signed_orientation(reflected) < 0
print("Signed orientations:", signed_orientation(points), signed_orientation(reflected))
print("Maximum pair-distance difference:", np.max(abs(distance_matrix(points)-distance_matrix(reflected))))

In [ ]:
fig = plt.figure(figsize=(9, 3.8), layout="constrained")
for panel, (coordinates, title) in enumerate([(points, "Labeled points"), (reflected, "Their mirror reflection")], start=1):
    ax = fig.add_subplot(1, 2, panel, projection="3d")
    for index in range(1, 4):
        pair = coordinates[[0, index]]
        ax.plot(*pair.T, color="0.5")
    for index, position in enumerate(coordinates):
        ax.scatter(*position, s=55, color=["#334155", "#2563eb", "#b45309", "#15803d"][index])
        ax.text(*(position+0.08), str(index))
    ax.set(xlim=(-1.5, 1.5), ylim=(-0.3, 2.3), zlim=(-0.3, 3.3),
           xlabel="x", ylabel="y", zlabel="", title=title)
    ax.text2D(1.03, 0.5, "z", transform=ax.transAxes, rotation=90)
    ax.set_box_aspect((3.0, 2.6, 3.6))
fig.suptitle("Geometric illustration, not molecular coordinates: equal distances, opposite orientation")
plt.show()

**Explain:** preserving every labeled pair distance still leaves a reflection ambiguity in three dimensions. For an ordinary reflection-invariant scalar energy this can be appropriate; for a stereosensitive endpoint in a chiral environment, the input and context must preserve the relevant distinction. A representation should match the scientific question, rather than retain or discard handedness by accident.

## 11.4.6. Connect representations to the prediction question

| Representation | Information supplied to learning | A question to check |
| --- | --- | --- |
| Descriptors or fixed fingerprints + MLP | Explicitly chosen summaries of the molecule | Which distinctions were lost before training? |
| Bond graph + message passing | Atom/bond attributes and a connectivity graph | Are stereo, ionic state, and long-range effects represented adequately? |
| SMILES tokens + a sequence/attention model | A string serialization, often including stereo symbols | Are equivalent SMILES kept in the same data split and handled consistently? |
| 3D geometry + a geometric network | Coordinates or geometric features for a specified structure | Which conformer/state is represented, and are symmetry transformations correct? |

Attention combines information using learned weights; it does not by itself establish molecular invariance, mechanistic explanation, or causal importance. The [original Transformer paper](https://proceedings.neurips.cc/paper/2017/hash/3f5ee243547dee91fbd053c1c4a845aa-Abstract.html) introduces an attention-based sequence architecture. Applying that architecture to molecules still requires an appropriate chemical encoding, data, and evaluation.

Pretraining can change the amount of task-specific data needed, but does not remove the need to examine training-data overlap, experimental labels, domain shift, uncertainty, and an untouched test set. Larger models do not automatically outperform simple baselines on small chemical datasets. This notebook downloads or evaluates no pretrained model and makes no benchmark ranking.

In [ ]:
record = {
    "scope": "Architecture and differentiability checks; no experimental property prediction",
    "torch_version": str(torch.__version__), "rdkit_version": rdBase.rdkitVersion,
    "smiles": smiles, "seed": 17, "dtype": "float64", "device": "cpu",
    "node_features": NODE_FEATURES, "bond_features": [str(b) for b in BOND_TYPES],
    "architecture": {"hidden": 8, "rounds": 2, "readout": "sum then biased linear"},
    "omitted_information": ["stereochemistry", "isotopes", "coordinates", "experimental conditions"],
    "oxygen_count_step": {"learning_rate": learning_rate,
                          "loss_before": float(loss_before.detach()), "loss_after": float(loss_after)},
    "checks_passed": ["node equivariance", "graph invariance", "disconnected batching",
                      "loss decreases after one step", "stereo encoding collision", "reflection distance equality"],
}
(OUT / "architecture_checks.json").write_text(json.dumps(record, indent=2)+"\n", encoding="utf-8")
print("Saved architecture definition and diagnostics to", OUT)

## Exercises and answers

1. Why do we store each bond twice? What would happen if the two directions were assigned to different molecular graphs?
2. Describe the difference between equivariance of atom embeddings and invariance of a molecular scalar. Why is an atom-index feature problematic?
3. Explain why concatenating all atom vectors in their current order does not provide an invariant molecular representation by itself.
4. How does increasing the number of message-passing rounds change the receptive field? Does that guarantee accurate chemistry?
5. Why does the oxygen-count loss decrease without establishing a useful predictive model? What is the correct baseline?
6. Will more training fix the enantiomer collision with the feature function unchanged? What else would a stereosensitive property task need?
7. A dataset has several SMILES and conformers for each compound. Which unit should be grouped before the train/test split, and how does the deployment question affect that choice?

<details><summary>Suggested answers</summary>

1. Each atom receives its neighbor's message. Cross-graph edges would let one molecule's representation depend on another molecule in the batch; the batch-consistency check would catch the resulting error for suitable examples.
2. Reordering the inputs reorders per-atom outputs but leaves a graph-level scalar unchanged. An index represents storage order and can change with no chemical change.
3. Concatenation preserves order. A suitable symmetric pooling operation, or another architecture enforcing the desired invariance, is needed.
4. Each round communicates one additional edge away. Finite-depth local summaries, omitted information, inadequate data, and model error can still limit performance.
5. It checks optimization of a differentiable function on the same few graphs. Exact atom counting has zero error and requires no learning; no unseen-data claim is made.
6. No: identical inputs force identical outputs here. Include appropriate stereochemical/context information and relevant labels, then evaluate generalization.
7. At minimum, prevent alternate representations of the same compound from leaking across splits. Grouping related scaffolds, reactions, measurement batches, or time periods may also be needed to match the intended use.

</details>

## From neural networks to Chapter 12

Chemical calculations and learned predictions both depend on representations, reference states, assumptions, and validation. A successful program run checks execution; independent physical or experimental evidence supports scientific claims. Keep the inputs, units, preprocessing, splits, model settings, and limitations with every reported result.

[Previous: Part 3](Chapter11_Part3.ipynb) · [Course contents](Readme.md) · [Chapters 10–11 review](docs/chapters10-11-review.md)

[Next: Chapter 12, Part 1: molecular graphs](Chapter12_Part1.ipynb)